# Taller - Camara en Vivo: Captura y Procesamiento de Video en Tiempo Real con YOLO

## Computacion Visual - Semana 11

Este cuaderno implementa un sistema interactivo de captura y procesamiento de video en tiempo real utilizando la webcam, filtros tradicionales de procesamiento de imagenes en OpenCV y deteccion de objetos basada en redes neuronales profundas con **YOLOv8**.

De acuerdo con las especificaciones del proyecto, **toda la interfaz se controla mediante botones, interruptores (toggles) y controles deslizantes (sliders)**, eliminando por completo el uso de controles por teclado y excluyendo el uso de caracteres emoji para mantener una estetica sobria, limpia y profesional.

### Objetivos del Taller
1. **Captura en Tiempo Real**: Configurar un flujo continuo de video utilizando OpenCV con el objetivo de alcanzar una tasa fluida (30+ FPS).
2. **Procesamiento de Filtros**: Aplicar transformaciones lineales y no lineales conmutables:
   - Escala de grises (`cv2.cvtColor`)
   - Binarización (umbralizacion simple de contraste)
   - Deteccion de bordes (`cv2.Canny`)
3. **Integracion de YOLOv8**: Cargar el modelo preentrenado YOLOv8 Nano para detectar objetos con cajas delimitadoras esteticas, etiquetas y porcentajes de confianza en tiempo real.
4. **Accion Condicional (Bonus)**: Activar un modo de alerta visual y filtro termico en el dashboard cuando se detecte una persona en el encuadre.
5. **Interfaz ipywidgets**: Construir un panel de control interactivo dentro de Jupyter utilizando `ipywidgets` y procesamiento asincrono multihilo.

### 1. Preparacion del Entorno e Instalacion de Dependencias

Para ejecutar este cuaderno se requieren librerias cientificas y de computacion visual. Ejecute la siguiente celda para verificar e instalar las dependencias necesarias en su entorno de Python.

In [7]:
import sys
!{sys.executable} -m pip install -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2. Marco Teorico del Procesamiento en Tiempo Real

#### Filtros de OpenCV Utilizados:
- **Escala de Grises**: Reduce la dimensionalidad del color de 3 canales (BGR) a 1 canal representativo de la luminancia ($Y = 0.299R + 0.587G + 0.114B$). Facilita el analisis estructural eliminando informacion redundante de crominancia.
- **Binarización**: Convierte una imagen a escala de grises en una imagen puramente binaria (blanco y negro) aplicando un umbral fijo o adaptativo. Es util para segmentacion de objetos con contrastes marcados.
- **Deteccion de Bordes Canny**: Algoritmo clasico multi-etapa que identifica transiciones de gradiente intensas en la imagen. Utiliza filtros gaussianos para suavizar el ruido, operadores Sobel para calcular gradientes de intensidad y umbralizacion por histeresis con dos limites (minimo y maximo) para conectar contornos.

#### Inferencia en Deep Learning (YOLOv8):
**YOLO (You Only Look Once)** es un detector de objetos de una sola etapa que reformula la deteccion como un problema de regresion espacial. A diferencia de las redes de dos etapas (como Faster R-CNN), YOLO procesa toda la imagen de una sola pasada, prediciendo simultaneamente multiples bounding boxes y sus probabilidades de clase. Esto permite tasas de procesamiento sumamente rapidas (30+ FPS en CPU de consumo basico con el modelo Nano `yolov8n.pt`).

### 3. Implementacion del Pipeline de Procesamiento de Imagenes

A continuacion se definen los modulos y clases que controlan la captura de video, los filtros tradicionales y el dibujado de la telemetria en pantalla.

In [8]:
import os
import time
import datetime
import urllib.request
import threading
import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from ultralytics import YOLO

# Asegurar carpetas de guardado
os.makedirs("../media", exist_ok=True)
os.makedirs("media", exist_ok=True)

def get_media_path(filename):
    if os.path.exists("../media"):
        return os.path.join("../media", filename)
    return os.path.join("media", filename)

def download_fallback_video():
    target_path = "people-detection.mp4"
    if os.path.exists(target_path):
        return target_path
    url = "https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/people-detection.mp4"
    print(f"No se detecto webcam fisica. Descargando video de prueba...")
    try:
        urllib.request.urlretrieve(url, target_path)
        print(f"Descarga completada.")
        return target_path
    except Exception as e:
        print(f"Error al descargar: {e}")
        return None

### 4. Implementacion del Procesador de Video Asincrono de Jupyter

La clase `JupyterYOLOProcessor` encapsula la ejecucion en un hilo de fondo (`threading.Thread`). Esto es crucial para evitar que el loop infinito de lectura de camara bloquee el entorno interactivo de Jupyter Notebook, permitiendo que los controles de `ipywidgets` actuen de manera reactiva al instante.

In [9]:
class JupyterYOLOProcessor:
    def __init__(self):
        self.model = YOLO("yolov8n.pt")
        self.display_width = 640
        self.display_height = 480
        
        # Estados de control vinculados a widgets
        self.active_filter = "Original"
        self.is_paused = False
        self.confidence_threshold = 0.50
        self.take_snapshot_flag = False
        self.is_recording = False
        
        # Hilo de ejecucion
        self.running = False
        self.thread = None
        self.video_writer = None
        self.cap = None
        
        # Controles y estadisticas de FPS
        self.fps_display = 0.0
        
        # Widget de renderizado de imagen
        self.image_widget = widgets.Image(format='jpeg', width=self.display_width, height=self.display_height)
        
    def apply_filters(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if self.active_filter == "Grayscale":
            return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        elif self.active_filter == "Binarization":
            _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
            return cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
        elif self.active_filter == "Canny":
            edges = cv2.Canny(gray, 50, 150)
            return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
        return frame

    def draw_custom_box(self, frame, x1, y1, x2, y2, label, conf, color):
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 1, cv2.LINE_AA)
        # Esquinas reforzadas
        lr = 10
        cv2.line(frame, (x1, y1), (x1 + lr, y1), color, 2, cv2.LINE_AA)
        cv2.line(frame, (x1, y1), (x1, y1 + lr), color, 2, cv2.LINE_AA)
        cv2.line(frame, (x2, y1), (x2 - lr, y1), color, 2, cv2.LINE_AA)
        cv2.line(frame, (x2, y1), (x2, y1 + lr), color, 2, cv2.LINE_AA)
        cv2.line(frame, (x1, y2), (x1 + lr, y2), color, 2, cv2.LINE_AA)
        cv2.line(frame, (x1, y2), (x1, y2 - lr), color, 2, cv2.LINE_AA)
        cv2.line(frame, (x2, y2), (x2 - lr, y2), color, 2, cv2.LINE_AA)
        cv2.line(frame, (x2, y2), (x2, y2 - lr), color, 2, cv2.LINE_AA)
        
        caption = f"{label} {conf*100:.0f}%"
        font = cv2.FONT_HERSHEY_SIMPLEX
        (tw, th), bl = cv2.getTextSize(caption, font, 0.4, 1)
        ly = y1 - 5 if y1 - 5 > th + 5 else y1 + th + 5
        cv2.rectangle(frame, (x1, ly - th - 2), (x1 + tw + 4, ly + bl), color, -1)
        cv2.putText(frame, caption, (x1 + 2, ly - 1), font, 0.4, (255, 255, 255), 1, cv2.LINE_AA)

    def draw_telemetry(self, frame, fps, counts, total, person_detected):
        overlay = frame.copy()
        # Barra superior traslucida
        cv2.rectangle(overlay, (0, 0), (self.display_width, 40), (25, 20, 18), -1)
        # Barra inferior traslucida
        cv2.rectangle(overlay, (0, self.display_height - 30), (self.display_width, self.display_height), (25, 20, 18), -1)
        if person_detected:
            cv2.rectangle(overlay, (0, self.display_height - 30), (self.display_width, self.display_height), (15, 10, 100), -1)
            if int(time.time() * 2) % 2 == 0:
                cv2.rectangle(frame, (0, 0), (self.display_width, self.display_height), (0, 0, 200), 5)
        cv2.addWeighted(overlay, 0.8, frame, 0.2, 0, frame)
        
        # Texto superior
        info_str = f"FILTRO: {self.active_filter.upper()} | FPS: {fps:.1f}"
        if self.is_recording:
            info_str += " | GRABANDO MULTIMEDIA"
        cv2.putText(frame, info_str, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (240, 240, 240), 1, cv2.LINE_AA)
        
        # Conteo inferior
        parts = [f"{cls.upper()}: {count}" for cls, count in counts.items()]
        counts_str = " | ".join(parts) if parts else "SIN DETECCIONES"
        total_str = f"OBJETOS DETECTADOS: {total} ({counts_str})"
        cv2.putText(frame, total_str, (10, self.display_height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (220, 220, 220), 1, cv2.LINE_AA)
        
        # Alerta intruso
        if person_detected and int(time.time() * 2.5) % 2 == 0:
            alert_txt = "ALERTA: PERSONA DETECTADA"
            asize = cv2.getTextSize(alert_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)[0]
            ax = (self.display_width - asize[0]) // 2
            cv2.rectangle(frame, (ax - 10, 5), (ax + asize[0] + 10, 35), (0, 0, 180), -1)
            cv2.putText(frame, alert_txt, (ax, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

    def start(self):
        if self.running:
            return
        self.running = True
        self.thread = threading.Thread(target=self.camera_loop)
        self.thread.daemon = True
        self.thread.start()
        
    def stop(self):
        self.running = False
        if self.thread:
            self.thread.join(timeout=2.0)
        if self.cap:
            self.cap.release()
        if self.video_writer:
            self.video_writer.release()
        self.image_widget.value = b''
        print("Camara cerrada y recursos liberados cleanly.")
        
    def camera_loop(self):
        self.cap = cv2.VideoCapture(0)
        using_fallback = False
        
        if not self.cap.isOpened():
            vpath = download_fallback_video()
            if vpath and os.path.exists(vpath):
                self.cap = cv2.VideoCapture(vpath)
                using_fallback = True
            else:
                print("No se pudo inicializar la captura de video.")
                self.running = False
                return
                
        prev_time = time.time()
        frame_count = 0
        last_frame_processed = None
        
        prev_counts = {}
        prev_total = 0
        prev_person = False
        
        while self.running:
            if self.is_paused and last_frame_processed is not None:
                # Re-dibujar HUD en pausa para actualizar FPS
                frame_to_show = last_frame_processed.copy()
                now = time.time()
                frame_count += 1
                if now - prev_time >= 1.0:
                    self.fps_display = frame_count / (now - prev_time)
                    frame_count = 0
                    prev_time = now
                
                self.draw_telemetry(frame_to_show, self.fps_display, prev_counts, prev_total, prev_person)
                _, jpeg = cv2.imencode('.jpg', frame_to_show)
                self.image_widget.value = jpeg.tobytes()
                time.sleep(0.05)
                continue
                
            ret, frame = self.cap.read()
            if not ret:
                if using_fallback:
                    self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                    continue
                else:
                    break
                    
            frame = cv2.resize(frame, (self.display_width, self.display_height))
            
            # Deteccion YOLOv8
            detections = []
            counts = {}
            total = 0
            person_detected = False
            
            results = self.model.predict(frame, verbose=False, conf=self.confidence_threshold)
            if results and len(results) > 0:
                result = results[0]
                for box in result.boxes:
                    cls_id = int(box.cls[0])
                    cls_name = self.model.names[cls_id]
                    conf = float(box.conf[0])
                    x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                    
                    detections.append((x1, y1, x2, y2, cls_name, conf))
                    counts[cls_name] = counts.get(cls_name, 0) + 1
                    total += 1
                    if cls_name == "person":
                        person_detected = True
                        
            # Almacenar para el estado de pausa
            prev_counts = counts
            prev_total = total
            prev_person = person_detected
            
            # Filtros y accion condicional termica
            if person_detected and self.active_filter in ["Grayscale", "Binarization", "Canny"]:
                base_filtered = self.apply_filters(frame.copy())
                gray_tmp = cv2.cvtColor(base_filtered, cv2.COLOR_BGR2GRAY)
                thermal = cv2.applyColorMap(gray_tmp, cv2.COLORMAP_JET)
                frame_to_show = cv2.addWeighted(thermal, 0.7, frame, 0.3, 0)
            else:
                frame_to_show = self.apply_filters(frame.copy())
                
            # Dibujar Bounding boxes de YOLO
            for x1, y1, x2, y2, label, conf in detections:
                color = (50, 220, 50) if label != "person" else (0, 0, 220)
                self.draw_custom_box(frame_to_show, x1, y1, x2, y2, label, conf, color)
                
            # Calcular FPS reales
            now = time.time()
            frame_count += 1
            if now - prev_time >= 1.0:
                self.fps_display = frame_count / (now - prev_time)
                frame_count = 0
                prev_time = now
                
            last_frame_processed = frame_to_show.copy()
            
            # Dibujar Telemetria y Alerta
            self.draw_telemetry(frame_to_show, self.fps_display, counts, total, person_detected)
            
            # Guardar Snapshot en el hilo
            if self.take_snapshot_flag:
                ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                fn = f"snapshot_{ts}.png"
                fp = get_media_path(fn)
                cv2.imwrite(fp, frame_to_show)
                print(f"Snapshot guardado en: {fp}")
                self.take_snapshot_flag = False
                
            # Escribir frame si se esta grabando
            if self.is_recording:
                if self.video_writer is None:
                    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                    vn = f"clip_{ts}.avi"
                    vp = get_media_path(vn)
                    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
                    self.video_writer = cv2.VideoWriter(vp, fourcc, 20.0, (self.display_width, self.display_height))
                    print(f"Iniciando grabacion de video en: {vp}")
                self.video_writer.write(frame_to_show)
            else:
                if self.video_writer is not None:
                    self.video_writer.release()
                    self.video_writer = None
                    print("Archivo de video guardado y cerrado.")
            
            # Convertir frame a JPEG para actualizar el widget de Jupyter
            _, jpeg = cv2.imencode('.jpg', frame_to_show)
            self.image_widget.value = jpeg.tobytes()
            
            # Descansar para regular FPS y no consumir 100% de CPU
            time.sleep(0.015)

### 5. Construccion del Dashboard Interactivo (`ipywidgets`)

Ahora configuramos los componentes de la interfaz de usuario. Al hacer clic sobre los toggles y botones se modifican las variables internas del procesador instantaneamente.

In [10]:
processor = JupyterYOLOProcessor()

# 1. Selectores de Filtro (Botones conmutadores)
filter_selector = widgets.ToggleButtons(
    options=['Original', 'Grayscale', 'Binarization', 'Canny'],
    description='Filtro Activo:',
    disabled=False,
    button_style='info',
    tooltips=['Ver imagen original con YOLO', 'Filtro escala de grises', 'Binarizar por umbral', 'Bordes Canny']
)

def change_filter(change):
    processor.active_filter = change['new']
filter_selector.observe(change_filter, names='value')

# 2. Control deslizante de Umbral de Confianza YOLO
conf_slider = widgets.FloatSlider(
    value=0.50,
    min=0.10,
    max=1.00,
    step=0.05,
    description='Confianza YOLO:',
    disabled=False,
    continuous_update=True,
    orientation='horizontal',
    readout=True,
    readout_format='.2f'
)

def change_conf(change):
    processor.confidence_threshold = change['new']
conf_slider.observe(change_conf, names='value')

# 3. Boton Pause / Play (Toggle)
btn_pause = widgets.ToggleButton(
    value=False,
    description='Pausar Video',
    disabled=False,
    button_style='warning',
    tooltip='Congelar el streaming de video'
)

def toggle_pause(change):
    processor.is_paused = change['new']
    btn_pause.description = 'Reanudar Video' if processor.is_paused else 'Pausar Video'
btn_pause.observe(toggle_pause, names='value')

# 4. Boton de Snapshot (Captura de pantalla)
btn_snapshot = widgets.Button(
    description='Capturar Imagen',
    disabled=False,
    button_style='primary',
    tooltip='Guardar el frame actual en carpeta media'
)

def trigger_snapshot(b):
    processor.take_snapshot_flag = True
    print("Capturando snapshot...")
btn_snapshot.on_click(trigger_snapshot)

# 5. Boton de Record (Grabacion de Video)
btn_record = widgets.ToggleButton(
    value=False,
    description='Grabar Clip',
    disabled=False,
    button_style='success',
    tooltip='Iniciar/Detener grabacion de video'
)

def toggle_record(change):
    processor.is_recording = change['new']
    if processor.is_recording:
        btn_record.description = 'Detener Grabacion'
        btn_record.button_style = 'danger'
    else:
        btn_record.description = 'Grabar Clip'
        btn_record.button_style = 'success'
btn_record.observe(toggle_record, names='value')

# 6. Boton de Cierre Seguro
btn_close = widgets.Button(
    description='Cerrar Camara',
    disabled=False,
    button_style='danger',
    tooltip='Liberar la webcam y terminar ejecucion'
)

def close_camera(b):
    processor.stop()
    btn_pause.disabled = True
    btn_snapshot.disabled = True
    btn_record.disabled = True
    btn_close.disabled = True
    filter_selector.disabled = True
    conf_slider.disabled = True
btn_close.on_click(close_camera)

### 6. Ejecucion de la Camara en Vivo

Ejecute la siguiente celda para iniciar la transmision de video y desplegar el panel de control interactivo. Puede cambiar los filtros, ajustar el control deslizante de confianza y guardar imagenes/videos de forma instantanea sin necesidad de usar el teclado.

In [11]:
# Desplegar Layout estético
control_panel = widgets.VBox([
    widgets.HTML("<h3>Panel de Control - Camara en Vivo</h3>"),
    filter_selector,
    conf_slider,
    widgets.HBox([btn_pause, btn_snapshot, btn_record, btn_close])
])

# Mostrar el video y los controles en paralelo
display(widgets.HBox([processor.image_widget, control_panel]))

# Iniciar el procesador de video asincrono
processor.start()